In [ ]:
import numpy as np
import pandas as pd
import data_utils_clean
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder,MinMaxScaler,OrdinalEncoder,PowerTransformer
from sklearn.model_selection import train_test_split

In [ ]:
!pip install dagshub

In [ ]:
import dagshub

dagshub.init(repo_owner='sneha12603',repo_name='Delivery-time-prediction',mlflow=True)

In [ ]:
!pip install mlflow

In [ ]:
import mlflow

In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/sneha12603/Delivery-time-prediction.mlflow")

In [ ]:
mlflow.set_experiment('Exp 6- Stacking Regresssor')


In [ ]:
from sklearn import set_config
set_config(transform_output='pandas')


In [ ]:
df = pd.read_csv('swiggy.csv')
df

In [ ]:
data_utils_clean.perform_data_cleaning(df)

In [ ]:
df = pd.read_csv('swiggy_cleaned.csv')
df

In [ ]:
df.columns

In [ ]:
#drop columns that is not required model input
columns_to_drop = ['rider_id',
                   'restaurant_latitude',
                   'restaurant_longitude',
                   'delivery_latitude',
                   'delivery_longitude',
                   'order_date',
                   'order_time_hour',
                   'order_day',
                   'city_name',
                   'order_day_of_week',
                   'order_month']

df.drop(columns=columns_to_drop,inplace=True)
df

In [ ]:
#check for missing values
df.isna().sum()

In [ ]:
#check for duplicates
df.duplicated().sum()

In [ ]:
import missingno as msno
msno.matrix(df)

In [ ]:
#columns that have missing values
missing_cols = (
    df.isna().any(axis=0).loc[lambda x:x].index

)

missing_cols

Drop missing values


In [ ]:
temp_df = df.copy().dropna()

In [ ]:
#split into X and y
X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

X

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print("The size of train data is",X_train.shape)
print("The size of test data is",X_test.shape)

In [ ]:
X_train.isna().sum()

In [ ]:
pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [ ]:
missing_cols

In [ ]:
#percentage of rows in data having missing values
(X_train.isna().any(axis=1).mean().round(2)*100)

Pre-Processing Pipeline

In [ ]:
num_cols = ['age','ratings','pickup_time_minutes','distance']

nominal_cat_cols = ['weather','type_of_order',
            'type_of_vehicle','festival',
            'city_type','is_weekend',
            'order_time_of_day']

ordinal_cat_cols = ['traffic','distance_type']

In [ ]:
nominal_cat_cols

In [ ]:
X_train.isna().sum()

In [ ]:
#do basic preprocessing
num_cols = ['age','ratings','pickup_time_minutes','distance']

nominal_cat_cols = ['weather','type_of_order',
                    'type_of_vehicle','festival','city_type',
                    'is_weekend','order_time_of_day']

ordinal_cat_cols = ['traffic','distance_type']

In [ ]:
#generate oorder for ordinal encoding
traffic_order = ['low','medium','high','jam']

distance_type_order = ['short','medium','long','very_long']


In [ ]:
#unique categories the ordinal columns
for col in ordinal_cat_cols:
  print(col,X_train[col].unique)

In [ ]:
#build a preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('scale',MinMaxScaler(),num_cols),
    ('nominal',OneHotEncoder(drop='first',handle_unknown='ignore',
                             sparse_output=False),nominal_cat_cols),
    ('ordinal_encode',OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                     encoded_missing_value=-999,
                                     handle_unknown='use_encoded_value',
                                     unknown_value=-1),ordinal_cat_cols)
],remainder='passthrough',n_jobs=-1,verbose_feature_names_out=False)

preprocessor

In [ ]:
#build the pipeline
processing_pipeline = Pipeline(steps=[
    ('preprocess',preprocessor),
])

processing_pipeline

In [ ]:
#do data preprocessing
X_train_trans = processing_pipeline.fit_transform(X_train)

X_test_trans = processing_pipeline.transform(X_test)

In [ ]:
X_train_trans

In [ ]:
!pip install optuna

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
import optuna
from sklearn.metrics import mean_absolute_error

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import StackingRegressor

In [ ]:
#build the best models
best_rf_params = {'n_estimators':150,
                  'criterion':'squared_error',
                  'max_depth':19,
                  'max_features': None,
                  'min_samples_split':2,
                  'min_samples_leaf':5,
                  'max_samples':0.7106001839374338

                  }
best_lgbm_params = {'n_estimators': 32,
                    'max_depth': 38,
                    'learning_rate': 0.19180137911248882,
                    'subsample': 0.7071645112302646,
                    'min_chile_weight': 7,
                    'min_split_gain': 0.005365828476410886,
                    'reg_lamda': 8.308142160805943}

best_rf =  RandomForestRegressor(**best_rf_params)
best_lgbm = LGBMRegressor(**best_lgbm_params)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
import optuna
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import StackingRegressor
from sklearn.compose import TransformedTargetRegressor

def objective(trial):
  with mlflow.start_run(nested=True):
    meta_model_name = trial.suggest_categorical("model",["LR","KNN","DT"])

    if meta_model_name == "LR":
      meta = LinearRegression()

    elif meta_model_name == "KNN":
      n_neighbors_knn = trial.suggest_int("n_neighbors_knn",1,15)
      weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
      meta = KNeighborsRegressor(n_neighbors=n_neighbors_knn,
                                 weights=weights_knn,n_jobs=-1)

    elif meta_model_name == "DT":
      max_depth_dt = trial.suggest_int("max_depth",1,10)
      min_samples_split_dt = trial.suggest_int("min_samples_split_dt",2,10)
      min_samples_leaf_dt = trial.suggest_int("min_samples_leaf_dt",1,10)
      meta = DecisionTreeRegressor(max_depth=max_depth_dt,
                                  min_samples_split = min_samples_split_dt,
                                  min_samples_leaf=min_samples_leaf_dt,
                                  random_state=42)
    #log meta model
    mlflow.log_param("meta_model_name",meta_model_name)

    #stacking regressor
    stacking_reg = StackingRegressor(estimators=[("rf",best_rf),
                                                 ("lgbm",best_lgbm)],
                                     final_estimator=meta,
                                     cv=5,n_jobs=-1)

    #build transformed regressor
    model = TransformedTargetRegressor(regressor=stacking_reg,transformer=pt)

    #train the model
    model.fit(X_train_trans,y_train)

    #get the predictions
    y_pred_test = model.predict(X_test_trans)

    #mean absolute error
    error = mean_absolute_error(y_test,y_pred_test)

    #log error
    mlflow.log_metric("MAE",error)

    return error

In [ ]:
#create optuna study
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name='best_model'):
  study.optimize(objective,n_trials=20,n_jobs=-1,show_progress_bar=True)

  mlflow.log_params(study.best_params)

  mlflow.log_metric('best_score',study.best_value)

In [ ]:
best_params = study.best_params
best_params

In [ ]:
study.trials_dataframe()['params_model'].value_counts()

In [ ]:
#mean scores for each meta estimaor type
study.trials_dataframe().groupby(by='params_model')['value'].mean().sort_values()

In [ ]:
#best score
study.best_value

In [ ]:
#optimization history plot
optuna.visualization.plot_optimization_history(study)

In [ ]:
#parallel coorrdinate
optuna.visualization.plot_parallel_coordinate(study,params=["model"])